# Bayesian Optimization

Companion notebook for the [Bayesian Optimization lesson](https://ml-viz.vercel.app/courses/bayesian-methods/03-bayesian-optimization).

We run the full BO loop on an expensive 1-D black box: a **GP surrogate**, the **UCB** and
**Expected Improvement** acquisition functions, and the adaptive loop that finds the optimum in a
handful of evaluations — far fewer than a grid would need. Pure NumPy + Matplotlib.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import erf, sqrt, pi
plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#444', 'axes.labelcolor': '#ccc',
    'xtick.color': '#888', 'ytick.color': '#888',
    'text.color': '#eee', 'grid.color': '#333', 'lines.linewidth': 2,
})
rng = np.random.default_rng(0)

## 1 — The expensive black box and a GP surrogate

We pretend `f` is expensive (in reality each call would train a model). The GP surrogate predicts f
everywhere with uncertainty from just a few evaluations.

In [ ]:
def f(x):                                   # the expensive black box (we want its max)
    return np.sin(3 * x) + 0.5 * x - 0.1 * x**2

def rbf_kernel(A, B, ell=0.7, sig=1.0):
    return sig**2 * np.exp(-((A[:, None] - B[None, :])**2) / (2 * ell**2))

def gp(Xtr, ytr, Xs, noise=1e-4):
    K = rbf_kernel(Xtr, Xtr) + noise * np.eye(len(Xtr))
    Ks = rbf_kernel(Xtr, Xs); Kss = rbf_kernel(Xs, Xs)
    Kinv = np.linalg.inv(K)
    mu = Ks.T @ Kinv @ ytr
    sd = np.sqrt(np.clip(np.diag(Kss - Ks.T @ Kinv @ Ks), 1e-9, None))
    return mu, sd

grid = np.linspace(-3, 3, 200)
print('true optimum near x =', round(grid[np.argmax(f(grid))], 2))

## 2 — Acquisition functions: UCB and EI

UCB = μ + κσ. EI = expected improvement over the current best. Both score points using the
surrogate's mean *and* uncertainty, trading exploitation against exploration.

In [ ]:
def norm_cdf(z): return 0.5 * (1 + np.vectorize(erf)(z / sqrt(2)))
def norm_pdf(z): return np.exp(-0.5 * z**2) / sqrt(2 * pi)

def ucb(mu, sd, kappa=2.0):
    return mu + kappa * sd

def expected_improvement(mu, sd, best, xi=0.01):
    imp = mu - best - xi
    z = imp / np.where(sd > 0, sd, 1)
    ei = imp * norm_cdf(z) + sd * norm_pdf(z)
    return np.where(sd > 0, ei, 0.0)

# with 3 initial points, where does each acquisition want to sample next?
Xtr = np.array([-2.5, 0.0, 2.5]); ytr = f(Xtr)
mu, sd = gp(Xtr, ytr, grid)
print('UCB picks x =', round(grid[np.argmax(ucb(mu, sd))], 2))
print('EI  picks x =', round(grid[np.argmax(expected_improvement(mu, sd, ytr.max()))], 2))

## 3 — The Bayesian optimization loop

Fit surrogate → maximize EI → evaluate f once → repeat. Watch it home in on the optimum in ~10
expensive evaluations.

In [ ]:
Xtr = np.array([-2.5, 2.5]); ytr = f(Xtr)
for step in range(10):
    mu, sd = gp(Xtr, ytr, grid)
    x_next = grid[np.argmax(expected_improvement(mu, sd, ytr.max()))]
    Xtr = np.append(Xtr, x_next); ytr = np.append(ytr, f(x_next))
best_x = Xtr[np.argmax(ytr)]
print(f'best x found = {best_x:.2f}, f = {ytr.max():.3f}')
print(f'true optimum = {grid[np.argmax(f(grid))]:.2f}, f = {f(grid).max():.3f}')
print(f'used {len(Xtr)} expensive evaluations total')

mu, sd = gp(Xtr, ytr, grid)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(grid, f(grid), '--', color='#555', label='true f (hidden)')
ax.fill_between(grid, mu-2*sd, mu+2*sd, color='#6366f1', alpha=0.3, label='GP ±2σ')
ax.plot(grid, mu, color='#818cf8', label='surrogate mean')
ax.scatter(Xtr, ytr, color='#2dd4bf', zorder=5, label='evaluations')
ax.set_title('Bayesian optimization concentrates evaluations near the optimum')
ax.legend(facecolor='#1a1d27', edgecolor='#444', fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## ✏️ Your turn

**Exercise.** Implement `ucb_acquisition(mu, sd, kappa)` (= μ + κσ) and
`propose_next(grid, mu, sd, kappa)` returning the grid point that maximizes UCB — the core of the BO
loop's 'where to sample next' decision.

In [ ]:
def ucb_acquisition(mu, sd, kappa=2.0):
    # TODO(you): upper confidence bound = mean + kappa * std
    return ...

def propose_next(grid, mu, sd, kappa=2.0):
    # TODO(you): return the grid point maximizing the UCB acquisition
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
mu, sd = gp(np.array([-2.5, 0.0, 2.5]), f(np.array([-2.5, 0.0, 2.5])), grid)
a = ucb_acquisition(mu, sd, 2.0)
assert np.allclose(a, mu + 2.0 * sd)
# larger kappa -> more exploration -> proposal moves toward higher-uncertainty regions
x_lo = propose_next(grid, mu, sd, kappa=0.1)
x_hi = propose_next(grid, mu, sd, kappa=5.0)
assert grid.min() <= x_hi <= grid.max()
assert propose_next(grid, mu, sd, 2.0) == grid[np.argmax(mu + 2.0 * sd)]
print(f'\u2713 UCB acquisition correct; exploit-pick={x_lo:.2f}, explore-pick={x_hi:.2f}')

<details>
<summary>Solution</summary>

```python
def ucb_acquisition(mu, sd, kappa=2.0):
    return mu + kappa * sd

def propose_next(grid, mu, sd, kappa=2.0):
    return grid[np.argmax(ucb_acquisition(mu, sd, kappa))]
```

κ is the explore/exploit dial: κ=0 is pure exploitation (sample the best predicted mean), large κ
weights the uncertainty term and explores. The acquisition function is cheap to optimize, so we
spend the expensive evaluation only on its chosen point.

</details>